# Notebook 05: Few-Shot Experiments (LoRA Adaptation)
## TSFM Industrial PdM Benchmark - ICATH Conference

**Author:** Yassire Ammouri  
**Purpose:** Run few-shot adaptation using LoRA on TSFMs  
**Method:** LoRA fine-tuning with 1% of training data  
**Expected Runtime:** 3-6 hours on T4 GPU

### Step 1: Setup and Imports

In [ ]:
import sys
from pathlib import Path
import torch
import numpy as np
import pandas as pd
import json
import time
from datetime import datetime
from tqdm import tqdm
import matplotlib.pyplot as plt
import yaml

# Add project source to path
PROJECT_DIR = Path("/content/tsfm-pdm-benchmark")
sys.path.insert(0, str(PROJECT_DIR / "src"))

from models import get_model, list_models
from evaluation.metrics import compute_all_metrics

# Paths
PROCESSED_DIR = PROJECT_DIR / "data/processed"
RESULTS_DIR = PROJECT_DIR / "results/few_shot"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

### Step 2: Configuration

In [ ]:
# Load configs
with open(PROJECT_DIR / "config/config.yaml", 'r') as f:
    config = yaml.safe_load(f)

few_shot_config = config['few_shot']

# Models that support few-shot (LoRA)
MODELS = ["moment", "patchtst"]  # Chronos doesn't support fine-tuning easily

# Datasets
DATASETS = ["cmapss/FD001", "cmapss/FD002"]

# Few-shot parameters
TRAIN_RATIO = few_shot_config['train_samples_ratio']  # 1%
LORA_R = few_shot_config['lora_r']  # 16
LORA_ALPHA = few_shot_config['lora_alpha']  # 32
EPOCHS = few_shot_config['epochs']  # 10
LR = few_shot_config['learning_rate']  # 1e-4

TASK = "forecasting"
HORIZON = config['preprocessing']['forecast_horizon']

print("Few-Shot Configuration:")
print(f"  Models: {MODELS}")
print(f"  Datasets: {DATASETS}")
print(f"  Train samples ratio: {TRAIN_RATIO*100:.1f}%")
print(f"  LoRA rank: {LORA_R}")
print(f"  LoRA alpha: {LORA_ALPHA}")
print(f"  Epochs: {EPOCHS}")
print(f"  Learning rate: {LR}")
print(f"  Total experiments: {len(MODELS) * len(DATASETS)}")

### Step 3: Few-Shot Experiment Function

In [ ]:
def run_few_shot(model_name, dataset_name, data_path, task="forecasting", 
                 horizon=96, device="cuda", train_ratio=0.01, 
                 lora_r=16, lora_alpha=32, epochs=10, lr=1e-4):
    """Run few-shot evaluation with LoRA adaptation"""
    print(f"\n{'='*60}")
    print(f"Few-Shot: {model_name} on {dataset_name}")
    print(f"{'='*60}")
    
    # Load data
    data = torch.load(data_path / "processed_data.pt")
    X_train = data['train_X']
    y_train = data['train_y']
    X_test = data['test_X']
    y_test = data['test_y']
    
    # Subsample training data (few-shot)
    n_train = max(int(len(X_train) * train_ratio), 32)
    X_train_sub = X_train[:n_train]
    y_train_sub = y_train[:n_train]
    
    print(f"Training samples ({train_ratio*100:.1f}%): {n_train}")
    print(f"Test samples: {len(X_test)}")
    
    # Load model
    model = get_model(model_name, device=device)
    model.load_model()
    
    # Few-shot adaptation
    print("\nRunning few-shot adaptation with LoRA...")
    start_time = time.time()
    
    model.few_shot_adapt(
        X_train_sub, y_train_sub,
        epochs=epochs,
        lr=lr,
        lora_r=lora_r,
        lora_alpha=lora_alpha
    )
    
    adaptation_time = time.time() - start_time
    print(f"Adaptation time: {adaptation_time:.2f}s")
    
    # Run inference
    print("\nRunning inference...")
    results = model.predict(X_test, horizon=horizon)
    predictions = results['predictions']
    
    # Handle shape mismatches
    if predictions.shape != y_test.shape:
        min_len = min(predictions.shape[1], y_test.shape[1])
        predictions = predictions[:, :min_len]
        y_test = y_test[:, :min_len]
        print(f"Shape adjusted to: {predictions.shape}")
    
    # Compute metrics
    metrics = compute_all_metrics(
        y_test.numpy() if isinstance(y_test, torch.Tensor) else y_test,
        predictions,
        task=task
    )
    
    print(f"Few-Shot Results: {metrics}")
    
    return {
        'model': model_name,
        'dataset': dataset_name,
        'task': task,
        'scenario': 'few_shot',
        'train_samples': n_train,
        'adaptation_time': adaptation_time,
        'metrics': metrics,
        'timestamp': datetime.now().isoformat(),
        'lora_r': lora_r,
        'lora_alpha': lora_alpha,
        'epochs': epochs,
        'lr': lr
    }

### Step 4: Run All Few-Shot Experiments

In [ ]:
all_results = []
errors = []

print(f"\nStarting few-shot experiments...")
print(f"Models: {MODELS}")
print(f"Datasets: {DATASETS}")
print(f"Total: {len(MODELS) * len(DATASETS)} experiments\n")

for dataset in tqdm(DATASETS, desc="Datasets"):
    dataset_path = PROCESSED_DIR / dataset
    
    if not dataset_path.exists():
        print(f"Warning: {dataset} not found, skipping")
        continue
    
    for model_name in tqdm(MODELS, desc=f"Models ({dataset})", leave=False):
        try:
            result = run_few_shot(
                model_name=model_name,
                dataset_name=dataset,
                data_path=dataset_path,
                task=TASK,
                horizon=HORIZON,
                device=device,
                train_ratio=TRAIN_RATIO,
                lora_r=LORA_R,
                lora_alpha=LORA_ALPHA,
                epochs=EPOCHS,
                lr=LR
            )
            all_results.append(result)
            
            # Save individual result
            safe_name = dataset.replace("/", "_")
            with open(RESULTS_DIR / f"{model_name}_{safe_name}_few_shot.json", 'w') as f:
                json.dump(result, f, indent=2)
            
        except Exception as e:
            print(f"Error with {model_name} on {dataset}: {e}")
            import traceback
            traceback.print_exc()
            errors.append({
                'model': model_name,
                'dataset': dataset,
                'error': str(e)
            })
    
    # Clear GPU memory
    torch.cuda.empty_cache()

print(f"\nExperiments complete!")
print(f"Successful: {len(all_results)}")
print(f"Errors: {len(errors)}")

### Step 5: Save Results

In [ ]:
# Create results DataFrame
df_results = pd.DataFrame(all_results)

# Save to CSV
df_results.to_csv(RESULTS_DIR / "few_shot_results.csv", index=False)
print(f"Results saved to: {RESULTS_DIR / 'few_shot_results.csv'}")

# Save errors
if errors:
    df_errors = pd.DataFrame(errors)
    df_errors.to_csv(RESULTS_DIR / "few_shot_errors.csv", index=False)
    print(f"Errors saved to: {RESULTS_DIR / 'few_shot_errors.csv'}")

### Step 6: Compare Zero-Shot vs Few-Shot

In [ ]:
print("=" * 80)
print("FEW-SHOT RESULTS SUMMARY")
print("=" * 80)

if len(df_results) > 0:
    # Extract metrics
    pivot_data = []
    for _, row in df_results.iterrows():
        pivot_data.append({
            'model': row['model'],
            'dataset': row['dataset'],
            'mae': row['metrics'].get('mae', float('nan')),
            'rmse': row['metrics'].get('rmse', float('nan')),
            'train_samples': row['train_samples']
        })
    
    df_few = pd.DataFrame(pivot_data)
    
    print("\nFew-Shot MAE:")
    print(df_few[['model', 'dataset', 'mae']].to_string(index=False))
    
    # Load zero-shot results for comparison
    zero_shot_file = PROJECT_DIR / "results/zero_shot/zero_shot_results.csv"
    if zero_shot_file.exists():
        df_zero = pd.read_csv(zero_shot_file)
        
        # Extract zero-shot metrics
        zero_data = []
        for _, row in df_zero.iterrows():
            try:
                metrics = json.loads(row['metrics'].replace("'", '"')) if isinstance(row['metrics'], str) else row['metrics']
                zero_data.append({
                    'model': row['model'],
                    'dataset': row['dataset'],
                    'mae_zero': metrics.get('mae', float('nan'))
                })
            except:
                pass
        
        df_zero_extract = pd.DataFrame(zero_data)
        
        # Merge
        df_compare = df_few.merge(df_zero_extract, on=['model', 'dataset'], how='left')
        df_compare['improvement'] = ((df_compare['mae_zero'] - df_compare['mae']) / df_compare['mae_zero'] * 100)
        
        print("\nZero-Shot vs Few-Shot Comparison:")
        print(df_compare[['model', 'dataset', 'mae_zero', 'mae', 'improvement']].to_string(index=False))
        
else:
    print("No results available!")

### Step 7: Visualize Results

In [ ]:
if len(df_results) > 0 and 'df_compare' in dir():
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Comparison bar chart
    x = np.arange(len(df_compare))
    width = 0.35
    
    axes[0].bar(x - width/2, df_compare['mae_zero'], width, label='Zero-Shot', color='#ff7f0e')
    axes[0].bar(x + width/2, df_compare['mae'], width, label='Few-Shot (1%)', color='#2ca02c')
    axes[0].set_xlabel('Experiment')
    axes[0].set_ylabel('MAE')
    axes[0].set_title('Zero-Shot vs Few-Shot MAE', fontsize=14, fontweight='bold')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels([f"{row['model']}\n{row['dataset']}" for _, row in df_compare.iterrows()], 
                            rotation=45, ha='right', fontsize=8)
    axes[0].legend()
    
    # Improvement percentage
    colors = ['#2ca02c' if imp > 0 else '#d62728' for imp in df_compare['improvement']]
    axes[1].bar(range(len(df_compare)), df_compare['improvement'], color=colors)
    axes[1].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    axes[1].set_xlabel('Experiment')
    axes[1].set_ylabel('Improvement (%)')
    axes[1].set_title('Few-Shot Improvement over Zero-Shot', fontsize=14, fontweight='bold')
    axes[1].set_xticks(range(len(df_compare)))
    axes[1].set_xticklabels([f"{row['model']}" for _, row in df_compare.iterrows()], 
                            rotation=45, ha='right', fontsize=8)
    
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "few_shot_comparison.png", dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"Figure saved to: {RESULTS_DIR / 'few_shot_comparison.png'}")

### Step 8: Save to Google Drive

In [ ]:
import shutil

DRIVE_RESULTS = Path("/content/drive/MyDrive/ICATH_Results/few_shot")
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)

# Copy results
for f in RESULTS_DIR.glob("*"):
    if f.is_file():
        shutil.copy(f, DRIVE_RESULTS / f.name)

print(f"Results backed up to: {DRIVE_RESULTS}")
print("\nNext: Run Notebook 06 (Cross-Domain Experiments)")